# Pre-finetune ImageNet-pretrained evaluation notebook

This notebook evaluates a timm model **before finetuning** on your dataset splits.

It does **not** train and does **not** save model checkpoints.  
You only need to change the top config cell: `DATA_ROOT`, `MODEL_KEY` / `MODEL_NAME`, and optionally `IMAGE_SIZE`, `BATCH_SIZE`, `EVAL_SPLITS`.

Important: if `NUM_CLASSES` is your dataset class count instead of 1000, timm loads ImageNet-pretrained weights where shapes match, but the classifier head is newly initialized for your dataset classes. This is a pre-finetune baseline, not a true 1000-class ImageNet-head evaluation unless you add a dataset-class to ImageNet-class mapping.


In [267]:
# Optional install cell for a fresh Vast.ai / Kaggle environment
# !pip install -q timm tqdm pillow torchvision pandas

In [269]:
from __future__ import annotations

import json
import os
import random
from collections import defaultdict
from contextlib import nullcontext
from pathlib import Path
from typing import Any, Dict

import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import timm
except ImportError as exc:
    raise ImportError("Missing dependency: timm. Install it with: pip install timm") from exc


## 1. Change only this config cell

In [270]:
# =========================
# EDIT THESE VALUES
# =========================

# Example:
# DATA_ROOT/
# ├── train/
# │   ├── region_1/
# │   │   ├── class_a/
# │   │   └── class_b/
# │   └── region_2/
# ├── val/
# └── test/
DATA_ROOT = Path("/kaggle/input/datasets/trieuvuongnguyen/dollarstreet/DollarStreet")

TRAIN_SPLIT = "train"
VAL_SPLIT = "val"
TEST_SPLIT = "test"

# Choose one from MODEL_CONFIGS below.
MODEL_KEY = "maxvit_base_224"

# Or set an exact timm model name here.
# If MODEL_NAME is not None, it overrides MODEL_KEY.
MODEL_NAME = None

# Evaluation settings
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
EVAL_SPLITS = ["val", "test"]   # options: ["val"], ["test"], or ["val", "test"]

# Pretrained ImageNet weights
PRETRAINED = True

# No model checkpoint is saved.
# This only saves JSON metrics, not the model.
SAVE_RESULTS = True
OUTPUT_DIR = Path("/kaggle/working/")

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]


In [271]:
MODEL_CONFIGS = {
    "convnextv2_base": "convnextv2_base.fcmae_ft_in22k_in1k",
    "swinv2_base": "swin_base_patch4_window7_224.ms_in22k_ft_in1k",
    "deit3_base": "deit3_base_patch16_224.fb_in22k_ft_in1k",
    "dinov2_vitb14": "vit_base_patch14_dinov2.lvd142m",
    "resnet50_base224": "resnet50.a2_in1k",
    "clip_vitb16_224": "vit_base_patch16_clip_224.openai_ft_in12k_in1k",
    "mae_vitb16_224": "vit_base_patch16_224.mae",
    "siglip_vitb16_224": "vit_base_patch16_siglip_224.webli",
    "maxvit_base_224": "maxvit_base_tf_224.in1k",
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

model_name = MODEL_NAME or MODEL_CONFIGS[MODEL_KEY]
device = torch.device(DEVICE)

print(f"DATA_ROOT:  {DATA_ROOT}")
print(f"MODEL:      {model_name}")
print(f"DEVICE:     {device}")
print(f"PRETRAINED: {PRETRAINED}")
print("SAVE_MODEL: False")


DATA_ROOT:  /kaggle/input/datasets/trieuvuongnguyen/dollarstreet/DollarStreet
MODEL:      maxvit_base_tf_224.in1k
DEVICE:     cuda
PRETRAINED: True
SAVE_MODEL: False


## 2. Utility functions

In [272]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def eval_transform(image_size: int = 224):
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
        ]
    )


def build_class_to_idx(root_path: str | Path) -> Dict[str, int]:
    root_path = Path(root_path)

    if not root_path.exists():
        raise FileNotFoundError(f"Class source path does not exist: {root_path}")

    class_names = set()

    # Expected: root/region/class/image
    for region in root_path.iterdir():
        if not region.is_dir():
            continue

        for category in region.iterdir():
            if category.is_dir():
                class_names.add(category.name)

    if not class_names:
        raise RuntimeError(f"No class folders found in {root_path}")

    return {class_name: idx for idx, class_name in enumerate(sorted(class_names))}


In [273]:
class MyDataset(Dataset):
    """
    Dataset for structure:

        split_root/
        ├── region_1/
        │   ├── class_a/
        │   └── class_b/
        └── region_2/
            ├── class_a/
            └── class_b/

    Returns:
        image, label, region
    """

    def __init__(self, root_path: str | Path, class_to_idx: Dict[str, int], transform=None):
        self.root_path = Path(root_path)
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.samples = []

        if not self.root_path.exists():
            raise FileNotFoundError(f"Dataset path does not exist: {self.root_path}")

        for region in sorted(self.root_path.iterdir()):
            if not region.is_dir():
                continue

            region_name = region.name

            for category in sorted(region.iterdir()):
                if not category.is_dir():
                    continue

                category_name = category.name
                if category_name not in self.class_to_idx:
                    continue

                label = self.class_to_idx[category_name]

                for img_path in sorted(category.iterdir()):
                    if not img_path.is_file():
                        continue
                    if img_path.suffix.lower() not in IMAGE_EXTENSIONS:
                        continue

                    self.samples.append(
                        {
                            "image_path": img_path,
                            "region": region_name,
                            "label": label,
                            "category": category_name,
                        }
                    )

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No images found under {self.root_path}. Expected split_root/region/class/image files."
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        sample = self.samples[idx]

        image = Image.open(sample["image_path"]).convert("RGB")
        label = sample["label"]
        region = sample["region"]

        if self.transform is not None:
            image = self.transform(image)

        return image, label, region


In [274]:
def build_eval_loader(split_dir: Path, class_to_idx: Dict[str, int]) -> DataLoader:
    dataset = MyDataset(
        root_path=split_dir,
        class_to_idx=class_to_idx,
        transform=eval_transform(IMAGE_SIZE),
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )

    print(f"{split_dir.name} samples: {len(dataset)}")
    return loader


def create_model_for_eval(model_name: str, num_classes: int, device: torch.device) -> nn.Module:
    model = timm.create_model(
        model_name,
        pretrained=PRETRAINED,
        num_classes=num_classes,
        drop_path_rate=0.0,
        # img_size=224,
    )

    model = model.to(device)
    model.eval()
    return model


def autocast_context(device: torch.device, use_amp: bool):
    if use_amp and device.type == "cuda":
        return torch.cuda.amp.autocast(enabled=True)
    return nullcontext()


In [275]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    split_name: str = "test",
) -> Dict[str, Any]:
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_images = 0

    region_correct = defaultdict(int)
    region_total = defaultdict(int)

    pbar = tqdm(loader, desc=f"Evaluate {split_name}", dynamic_ncols=True)

    for images, labels, regions in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast_context(device, USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        preds = logits.argmax(dim=1)
        correct_mask = preds == labels
        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += correct_mask.sum().item()
        total_images += batch_size

        for region, is_correct in zip(regions, correct_mask.detach().cpu().tolist()):
            region = str(region)
            region_total[region] += 1
            region_correct[region] += int(is_correct)

        pbar.set_postfix({"object_acc": f"{total_correct / max(total_images, 1):.4f}"})

    object_acc = total_correct / max(total_images, 1)
    avg_loss = total_loss / max(total_images, 1)

    region_object_acc = {
        region: region_correct[region] / region_total[region]
        for region in sorted(region_total.keys())
    }

    if region_object_acc:
        best_region = max(region_object_acc, key=region_object_acc.get)
        worst_region = min(region_object_acc, key=region_object_acc.get)
        best_region_acc = region_object_acc[best_region]
        worst_region_acc = region_object_acc[worst_region]
        region_bias_gap = best_region_acc - worst_region_acc
    else:
        best_region = None
        worst_region = None
        best_region_acc = None
        worst_region_acc = None
        region_bias_gap = None

    return {
        "loss": avg_loss,
        "object_acc": object_acc,
        "region_object_acc": region_object_acc,
        "best_region": best_region,
        "worst_region": worst_region,
        "best_region_acc": best_region_acc,
        "worst_region_acc": worst_region_acc,
        "region_bias_gap": region_bias_gap,
    }


## 3. Build dataset and model

In [276]:
seed_everything(SEED)

train_dir = DATA_ROOT / TRAIN_SPLIT
val_dir = DATA_ROOT / VAL_SPLIT
test_dir = DATA_ROOT / TEST_SPLIT

if not train_dir.exists():
    raise FileNotFoundError(f"Train split not found: {train_dir}")
if "val" in EVAL_SPLITS and not val_dir.exists():
    raise FileNotFoundError(f"Val split not found: {val_dir}")
if "test" in EVAL_SPLITS and not test_dir.exists():
    raise FileNotFoundError(f"Test split not found: {test_dir}")

class_to_idx = build_class_to_idx(train_dir)
idx_to_class = {int(v): k for k, v in class_to_idx.items()}
num_classes = len(class_to_idx)

print(f"Number of classes: {num_classes}")
print("Class mapping:")
for class_name, idx in class_to_idx.items():
    print(f"  {idx:>2}: {class_name}")

if PRETRAINED and num_classes != 1000:
    print(
        "\nNote: num_classes != 1000. The ImageNet-pretrained backbone is loaded where possible, "
        "but the final classifier head is initialized for your dataset classes."
    )


Number of classes: 64
Class mapping:
   0: 409_analog clock
   1: 412_trash can
   2: 418_ballpoint
   3: 428_wheelbarrow
   4: 453_bookcase
   5: 477_tool kit
   6: 487_cellphone
   7: 487_mobile phone
   8: 527_desktop computer
   9: 529_diaper
  10: 531_digital watch
  11: 532_dining table
  12: 533_dishrag
  13: 534_dishwasher
  14: 545_electric fan
  15: 567_skillet
  16: 623_paper knife
  17: 626_igniter
  18: 632_speaker
  19: 648_medicine cabinet
  20: 660_manufactured home
  21: 665_moped
  22: 669_mosquito net
  23: 671_all-terrain bike
  24: 679_necklace
  25: 695_padlock
  26: 700_paper towel
  27: 704_parking meter
  28: 705_passenger car
  29: 729_plate rack
  30: 737_soda bottle
  31: 754_radio
  32: 760_icebox
  33: 760_refrigerator
  34: 762_eating place
  35: 762_restaurant
  36: 765_rocking chair
  37: 770_running shoe
  38: 773_salt shaker
  39: 794_shower curtain
  40: 804_soap dispenser
  41: 811_space heater
  42: 813_spatula
  43: 827_stove
  44: 831_day bed
  4

In [277]:
loaders = {}

if "val" in EVAL_SPLITS:
    loaders["val"] = build_eval_loader(val_dir, class_to_idx)

if "test" in EVAL_SPLITS:
    loaders["test"] = build_eval_loader(test_dir, class_to_idx)

criterion = nn.CrossEntropyLoss()

model = create_model_for_eval(
    model_name=model_name,
    num_classes=num_classes,
    device=device,
)

print("\nModel loaded for evaluation.")
print("No optimizer created.")
print("No training loop will run.")
print("No model checkpoint will be saved.")


val samples: 4309
test samples: 4308

Model loaded for evaluation.
No optimizer created.
No training loop will run.
No model checkpoint will be saved.


## 4. Run pre-finetune evaluation

In [278]:
all_results = {
    "model_name": model_name,
    "pretrained": PRETRAINED,
    "data_root": str(DATA_ROOT),
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_classes": num_classes,
    "class_to_idx": class_to_idx,
    "idx_to_class": idx_to_class,
    "note": (
        "Pre-finetune evaluation only. No optimizer step, no training, no model checkpoint saving. "
        "If num_classes != 1000, this evaluates ImageNet-pretrained weights where shapes match "
        "plus a newly initialized dataset classifier head."
    ),
    "splits": {},
}

for split_name, loader in loaders.items():
    results = evaluate(
        model=model,
        loader=loader,
        criterion=criterion,
        device=device,
        split_name=split_name,
    )
    all_results["splits"][split_name] = results

    print(f"\n{split_name.upper()} RESULTS")
    print(f"Loss:             {results['loss']:.4f}")
    print(f"Object Acc:       {results['object_acc']:.4f}")

    if results["region_bias_gap"] is not None:
        print(f"Best Region:      {results['best_region']} ({results['best_region_acc']:.4f})")
        print(f"Worst Region:     {results['worst_region']} ({results['worst_region_acc']:.4f})")
        print(f"Region Bias Gap:  {results['region_bias_gap']:.4f}")

if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    result_path = OUTPUT_DIR / "pretrained_imagenet_eval_results.json"
    save_json(all_results, result_path)
    print(f"\nSaved JSON results to: {result_path}")
else:
    print("\nSAVE_RESULTS=False, so metrics were not written to disk.")


Evaluate val:   0%|          | 0/135 [00:00<?, ?it/s]

/tmp/ipykernel_58/2991138498.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast(enabled=True)



VAL RESULTS
Loss:             4.1524
Object Acc:       0.0285
Best Region:      am (0.0366)
Worst Region:     af (0.0230)
Region Bias Gap:  0.0136


Evaluate test:   0%|          | 0/135 [00:00<?, ?it/s]


TEST RESULTS
Loss:             4.1550
Object Acc:       0.0288
Best Region:      eu (0.0452)
Worst Region:     as (0.0198)
Region Bias Gap:  0.0254

Saved JSON results to: /kaggle/working/pretrained_imagenet_eval_results.json


## 5. Summary table

In [279]:
summary_rows = []

for split_name, results in all_results["splits"].items():
    summary_rows.append(
        {
            "split": split_name,
            "loss": results["loss"],
            "object_acc": results["object_acc"],
            "best_region": results["best_region"],
            "best_region_acc": results["best_region_acc"],
            "worst_region": results["worst_region"],
            "worst_region_acc": results["worst_region_acc"],
            "region_bias_gap": results["region_bias_gap"],
        }
    )

if pd is not None:
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
else:
    for row in summary_rows:
        print(row)


,split,loss,object_acc,best_region,best_region_acc,worst_region,worst_region_acc,region_bias_gap
0,val,4.152382,0.028545,am,0.036643,af,0.023030,0.013613
1,test,4.154955,0.028784,eu,0.045190,as,0.019802,0.025388


## 6. Region accuracy table

In [280]:
region_rows = []

for split_name, results in all_results["splits"].items():
    for region, acc in results["region_object_acc"].items():
        region_rows.append(
            {
                "split": split_name,
                "region": region,
                "object_acc": acc,
            }
        )

if pd is not None:
    region_df = pd.DataFrame(region_rows)
    display(region_df)
else:
    for row in region_rows:
        print(row)


,split,region,object_acc
0,val,af,0.023030
1,val,am,0.036643
2,val,as,0.024936
3,val,eu,0.035661
4,test,af,0.022432
5,test,am,0.042056
6,test,as,0.019802
7,test,eu,0.045190
